<a href="https://colab.research.google.com/github/Leeshleeshma/Medical-Image-Classification/blob/main/%20%20%20notebooks/06_build_nih_external_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 - Build the NIH ChestX-ray14 external test set (Google Colab)
**Group 5, ICSI 535.** Builds the cross-dataset test set in the same format as `processed_224_pad_gray` (padded 224 x 224 grayscale, `images` uint8 + `labels` int64, CSV with the same columns).

**Before you start (2 minutes):**
1. Kaggle account -> Settings -> API Tokens -> *Generate*. Copy the long token string (it is shown once).
2. Run the cells **top to bottom** (Runtime -> Run all). It takes roughly 30 to 60 minutes, mostly the 45 GB download.

**Result:** `535_project/data/processed_224_pad_gray_nih/` in your Google Drive with `test.npz`, `test.csv` and `reports/nih_summary.csv`.

## 1. Connect Google Drive and check disk space

In [1]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/535_project"   # same folder name as the group notebooks
import os
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
!df -h /content | tail -1
print("Need roughly 100 GB free on /content for the full NIH download. If 'Avail' is much lower, tell Claude/the group.")

Mounted at /content/drive
overlay         108G   21G   88G  20% /
Need roughly 100 GB free on /content for the full NIH download. If 'Avail' is much lower, tell Claude/the group.


## 2. Kaggle login (paste your API token when asked)

In [2]:
!pip -q install -U kagglehub
import os, shutil, getpass
tok = getpass.getpass("Paste your Kaggle API token (the long string from Settings -> API Tokens), or just press Enter to upload kaggle.json instead: ")
if tok.strip():
    os.environ["KAGGLE_API_TOKEN"] = tok.strip()
else:
    from google.colab import files
    files.upload()                           # choose kaggle.json
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle login ready.")

Paste your Kaggle API token (the long string from Settings -> API Tokens), or just press Enter to upload kaggle.json instead: ··········
Kaggle login ready.


## 3. Get the shared group code (`xray_data.py`) from GitHub

In [3]:
!rm -rf /content/Medical-Image-Classification
!git clone https://github.com/Leeshleeshma/Medical-Image-Classification /content/Medical-Image-Classification
!ls /content/Medical-Image-Classification/notebooks
!mkdir -p /content/Medical-Image-Classification/scripts

Cloning into '/content/Medical-Image-Classification'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 35 (delta 7), reused 9 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 1.52 MiB | 22.91 MiB/s, done.
Resolving deltas: 100% (7/7), done.
01_dataset_exploration.ipynb  04_patient_split_padded_resize.ipynb
02_dense_baseline.ipynb       05_resnet_transfer_learning.ipynb
03_cnn_from_scratch.ipynb     xray_data.py


## 4. Save the NIH build script

In [4]:
%%writefile /content/Medical-Image-Classification/scripts/make_nih_dataset.py
"""Build the NIH ChestX-ray14 external test set for Group 5 (ICSI 535), in the SAME format as
Ariana's data/processed_224_pad_gray (notebook 04 / xray_data.py).

Run from the project root (the folder with data/ and notebooks/):
    python make_nih_dataset.py --nih-dir /path/to/NIH

--nih-dir is the folder you downloaded from Kaggle ("NIH Chest X-rays", or the smaller
"sample" version). It must contain the label CSV (Data_Entry_2017*.csv or sample_labels.csv)
and the .png files, in any sub-folders. Images missing from the folder are skipped.

Output: data/processed_224_pad_gray_nih/test.npz and test.csv (+ reports/nih_summary.csv)
  - test.npz: images uint8 (N,224,224), labels int64 (0 = NORMAL, 1 = PNEUMONIA)
  - test.csv: same columns as Ariana's CSVs (path,label,class_name,patient_id,split,
    original_split,npz_index,width,height) + finding_labels, patient_age, patient_gender,
    view_position at the end. Row i of the CSV = image i of the npz.
Use it with the unchanged shared loader (it is an EXTERNAL test set, never trained on):
    NIH_DIR = PROJECT_DIR / "data" / "processed_224_pad_gray_nih"
    X_nih, y_nih = load_split("test", NIH_DIR, model="cnn")
    nih_ds = make_dataset("test", NIH_DIR, model="resnet")

Label rule: PNEUMONIA = "Pneumonia" is among the findings (may co-occur with others).
NORMAL = the only finding is "No Finding", and the patient has NO pneumonia image anywhere.
Default: at most 1 image per patient, and as many NORMAL as PNEUMONIA images (--neg-per-pos 1).
"""
import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--nih-dir", required=True)
    ap.add_argument("--project-dir", default=".")
    ap.add_argument("--neg-per-pos", type=float, default=1.0, help="NORMAL images per PNEUMONIA image")
    ap.add_argument("--max-per-patient", type=int, default=1, help="0 = no limit")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    project = Path(args.project_dir).resolve()
    nih = Path(args.nih_dir).resolve()
    for p in [project / "notebooks", project, Path.cwd()]:
        sys.path.insert(0, str(p))
    from xray_data import CLASS_NAMES, IMAGE_SIZE, load_image, load_npz  # shared code, unchanged

    out_dir = project / "data" / "processed_224_pad_gray_nih"
    (out_dir / "reports").mkdir(parents=True, exist_ok=True)

    # 1. Labels
    csvs = sorted(nih.rglob("Data_Entry_2017*.csv")) + sorted(nih.rglob("sample_labels.csv"))
    if not csvs:
        sys.exit(f"No Data_Entry_2017*.csv or sample_labels.csv found under {nih}")
    meta = pd.read_csv(csvs[0])
    print("labels file:", csvs[0], "|", len(meta), "rows,", meta["Patient ID"].nunique(), "patients")

    # 2. Which images do we actually have?
    on_disk = {p.name: p for p in nih.rglob("*.png")}
    meta["has_image"] = meta["Image Index"].isin(on_disk)
    print("images found on disk:", int(meta["has_image"].sum()))

    findings = meta["Finding Labels"].str.split("|")
    meta["is_pneu"] = findings.map(lambda f: "Pneumonia" in f)
    meta["is_normal"] = findings.map(lambda f: f == ["No Finding"])
    pneu_patients = set(meta.loc[meta["is_pneu"], "Patient ID"])        # judged on the full CSV
    meta["age"] = pd.to_numeric(meta["Patient Age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")

    def one_per_patient(df, shuffle_seed=None):
        if shuffle_seed is not None:
            df = df.sample(frac=1, random_state=shuffle_seed)
        else:
            df = df.sort_values(["Patient ID", "Follow-up #"])
        if args.max_per_patient > 0:
            df = df.groupby("Patient ID", sort=False).head(args.max_per_patient)
        return df

    pos = one_per_patient(meta[meta.has_image & meta.is_pneu])
    neg_pool = one_per_patient(
        meta[meta.has_image & meta.is_normal & ~meta["Patient ID"].isin(pneu_patients)], args.seed)
    n_neg = min(len(neg_pool), int(round(len(pos) * args.neg_per_pos)))
    if n_neg < len(pos) * args.neg_per_pos:
        print(f"WARNING: only {len(neg_pool)} NORMAL images available for {len(pos)} PNEUMONIA")
    neg = neg_pool.sample(n=n_neg, random_state=args.seed)
    if len(pos) == 0:
        sys.exit("No pneumonia images found in the downloaded folder (the sample set has very few).")

    # 3. Build the CSV (same columns as the shared splits, sorted by path, row = npz index)
    rows = []
    for label, d in [(0, neg), (1, pos)]:
        for r in d.itertuples(index=False):
            path = on_disk[r[meta.columns.get_loc("Image Index")]]
            with Image.open(path) as im:
                w, h = im.size
            rows.append({
                "path": path.relative_to(nih).as_posix(), "label": label,
                "class_name": CLASS_NAMES[label], "patient_id": f"NIH-{r[meta.columns.get_loc('Patient ID')]}",
                "split": "test", "original_split": "nih", "width": w, "height": h,
                "finding_labels": r[meta.columns.get_loc("Finding Labels")],
                "patient_age": r[meta.columns.get_loc("age")],
                "patient_gender": r[meta.columns.get_loc("Patient Gender")],
                "view_position": r[meta.columns.get_loc("View Position")],
            })
    df = pd.DataFrame(rows).sort_values("path").reset_index(drop=True)
    df["npz_index"] = np.arange(len(df))
    cols = ["path", "label", "class_name", "patient_id", "split", "original_split", "npz_index",
            "width", "height", "finding_labels", "patient_age", "patient_gender", "view_position"]
    df = df[cols]
    df.to_csv(out_dir / "test.csv", index=False)

    # 4. Images: the SAME pad_and_resize as the Kaggle data (via xray_data.load_image)
    images = np.empty((len(df), IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
    for i, rel in enumerate(df["path"]):
        images[i] = load_image(nih / rel)
        if i % 500 == 0:
            print(f"  {i}/{len(df)}")
    np.savez_compressed(out_dir / "test.npz", images=images, labels=df["label"].to_numpy(np.int64))

    # 5. Checks and summary
    assert df["patient_id"].is_unique or args.max_per_patient != 1, "patient repeated"
    raw_images, raw_labels = load_npz("test", out_dir)
    assert raw_images.shape == (len(df), IMAGE_SIZE, IMAGE_SIZE) and raw_images.dtype == np.uint8
    assert np.array_equal(raw_labels, df["label"].to_numpy()) and raw_labels.dtype == np.int64
    assert not set(df.loc[df.label == 0, "patient_id"]) & set(df.loc[df.label == 1, "patient_id"])
    summary = pd.DataFrame([{
        "split": "nih_test", "images": len(df), "patients": df["patient_id"].nunique(),
        "NORMAL": int((df.label == 0).sum()), "PNEUMONIA": int((df.label == 1).sum()),
        "% NORMAL": round(100 * (df.label == 0).mean(), 1),
        "median_age": df["patient_age"].median(), "pct_PA_view": round(100 * (df.view_position == "PA").mean(), 1),
    }])
    summary.to_csv(out_dir / "reports" / "nih_summary.csv", index=False)
    print("\n", summary.to_string(index=False))
    print("\nSaved to", out_dir, "| images", images.shape, images.dtype)
    print("SLIDE: NIH ChestX-ray14 external test set:",
          f"{len(df)} adult X-rays (median age {df['patient_age'].median():.0f}), "
          f"{int((df.label == 1).sum())} pneumonia / {int((df.label == 0).sum())} normal, 1 image per patient, "
          "same padding + 224x224 preprocessing as the Kaggle data.")


if __name__ == "__main__":
    main()

Writing /content/Medical-Image-Classification/scripts/make_nih_dataset.py


## 5. Download the full NIH ChestX-ray14 dataset (about 45 GB, the slow step)

In [5]:
import os
os.environ["KAGGLEHUB_CACHE"] = "/content/kh"
import kagglehub
NIH_DIR = kagglehub.dataset_download("nih-chest-xrays/data")
print("NIH folder:", NIH_DIR)
!df -h /content | tail -1

100%|██████████| 42.0G/42.0G [06:58<00:00, 108MB/s]

Extracting files...


NIH folder: /content/kh/datasets/nih-chest-xrays/data/versions/3
overlay         108G   63G   45G  59% /


## 6. Build the external test set

In [6]:
!cd /content/Medical-Image-Classification && python scripts/make_nih_dataset.py --nih-dir "{NIH_DIR}" --project-dir /content/Medical-Image-Classification

labels file: /content/kh/datasets/nih-chest-xrays/data/versions/3/Data_Entry_2017.csv | 112120 rows, 30805 patients
images found on disk: 112120
  0/2016
  500/2016
  1000/2016
  1500/2016
  2000/2016

    split  images  patients  NORMAL  PNEUMONIA  % NORMAL  median_age  pct_PA_view
nih_test    2016      2016    1008       1008      50.0        47.0         66.3

Saved to /content/Medical-Image-Classification/data/processed_224_pad_gray_nih | images (2016, 224, 224) uint8
SLIDE: NIH ChestX-ray14 external test set: 2016 adult X-rays (median age 47), 1008 pneumonia / 1008 normal, 1 image per patient, same padding + 224x224 preprocessing as the Kaggle data.


## 7. Copy the results to Google Drive (small files, safe from Colab resets)

In [7]:
import shutil
SRC = "/content/Medical-Image-Classification/data/processed_224_pad_gray_nih"
DST = DRIVE_PROJECT_DIR + "/data/processed_224_pad_gray_nih"
shutil.copytree(SRC, DST, dirs_exist_ok=True)
!ls -lh "{DST}"
print("\nDone. Paste the table and the SLIDE: line from step 6 to Claude and the group.")

total 71M
drwx------ 2 root root 4.0K Oct 25 08:24 reports
-rw------- 1 root root 205K Oct 25 08:24 test.csv
-rw------- 1 root root  71M Oct 25 08:24 test.npz

Done. Paste the table and the SLIDE: line from step 6 to Claude and the group.
